# Customer Analysis & Sales Insights

##  Business Questions

- Which customers provide the highest value to the company?
- Which customers may be at risk of not making future purchases?
- Which cities are the most suitable for advertising?
- Are Discounts actually useful?
- What actions should the company take to increase sales?

## Dataset Overview

The dataset contains customer level information, including demographic characteristics, purchase behavior, spending patterns, and customer satisfaction.

After data cleaning, the dataset contains 60 unique customers and 17 columns.

This dataset will be analyzed to understand customer value and behavior, identify potentially at-risk customers, evaluate discount effectiveness, identify suitable cities for advertising, and develop actionable strategies for increasing sales.

## Tools & Libraries

* Python
* Pandas
* Matplotlib
* Plotly

## Workflow


1. Data Loading & Initial Inspection**
2. Exploratory Data Analysis**
3. Customer Value Analysis**
4. Customer Purchase Risk Analysis**
5. City Level Advertising Analysis**
6. Discount Effectiveness Analysis**
7. Customer Segmentation Analysis**

   * Membership Tier
   * Age Group
   * Device & Payment Method
8. Business Insights & Recommendations**
9. Conclusion**
 

 ## Data Loading & Initial Inspection

In [1]:
#Import_Libraries

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
#load_dataset

Data = pd.read_csv("../data/cleaned_dataset_fatemeyousefia.csv")

#dataframe

df = pd.DataFrame(Data)

### Dataset Preview

In [3]:
Data.head(5)

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
0,1001,Reza,M,19,Karaj,Alborz,2025-02-19,VIP,17,121.53,2066.01,16,Card,Android,Yes,3,5
1,1002,Sina,M,53,Tehran,Tehran,2022-08-19,Gold,12,326.47,3917.64,3,Card,Web,No,5,3
2,1003,Parsa,M,31,Shiraz,Fars,2023-06-20,Gold,21,59.46,1248.66,22,Online_Wallet,iPhone,Yes,6,1
3,1004,Sina,M,58,Mashhad,Khorasan,2021-11-08,Gold,23,266.15,6121.45,40,Card,Android,No,4,4
4,1005,Kimia,F,28,Isfahan,Isfahan,2021-10-21,Silver,23,169.54,3899.42,273,Online_Wallet,Android,Yes,7,4


## Exploratory Data Analysis

The exploratory analysis focuses on customer value, purchase risk, geographic performance, discount usage, and customer segmentation. 

Each business question is analyzed using relevant customer level metrics and visualizations to identify actionable patterns.

### Question 1 : Which customers provide the highest value to the company?


In [4]:
# AOV
df['avg_order_value'].describe()

count     60.000000
mean     213.156500
std      130.509923
min       27.630000
25%      108.137500
50%      165.435000
75%      324.107500
max      449.810000
Name: avg_order_value, dtype: float64

In [5]:
# Top 10 AOV
top_10_aov = df.nlargest(10, 'avg_order_value')[['customer_id', 'avg_order_value']]

top_10_aov

,customer_id,avg_order_value
55,1056,449.81
56,1057,436.54
43,1044,434.98
11,1012,434.50
21,1022,423.43
53,1054,410.16
24,1025,406.03
7,1008,389.58
36,1037,387.68
9,1010,381.13


In [6]:
# Normalize the three metrics
metrics = ['avg_order_value', 'total_spending', 'purchase_count']

df['value_score'] = (
    (df['avg_order_value'] - df['avg_order_value'].min()) /
    (df['avg_order_value'].max() - df['avg_order_value'].min())
    +
    (df['total_spending'] - df['total_spending'].min()) /
    (df['total_spending'].max() - df['total_spending'].min())
    +
    (df['purchase_count'] - df['purchase_count'].min()) /
    (df['purchase_count'].max() - df['purchase_count'].min())
)

# Identify the top 10 customers
top_10 = df.nlargest(10, 'value_score')['customer_id']

# Create the plot
df['customer_group'] = df['customer_id'].apply( lambda x: 'Top 10' if x in top_10.values else 'Other')

fig = px.scatter(
    df,
    x='avg_order_value',
    y='total_spending',
    size='purchase_count',
    color='customer_group',
    hover_name='customer_id',
    hover_data={
        'avg_order_value': ':.2f',
        'total_spending': ':.2f',
        'purchase_count': True,
        'value_score': ':.2f'
    },
    text=df['customer_id'].where(df['customer_id'].isin(top_10)),
    title='Customer Value Analysis',
    labels={
        'avg_order_value': 'Average Order Value',
        'total_spending': 'Total Spending',
        'purchase_count': 'Purchase Count'
    }
)

fig.update_traces(
    textposition='top center')

fig.show()

To identify customers with strong performance across multiple purchasing dimensions, a composite value score was created using `avg_order_value`, `total_spending`, and `purchase_count`.

Since these metrics have different scales, each metric was normalized to a 0 – 1 range before being combined.

The three normalized metrics were equally weighted, and the resulting score was used to identify the top 10 customers with the strongest combined performance.

The scatter plot shows `avg_order_value` on the x-axis and `total_spending` on the y-axis, while the size of each point represents `purchase_count`.

In [7]:
#Top_10_customers

df[df['customer_id'].isin(top_10)][
    ['customer_id', 'avg_order_value',
     'total_spending', 'purchase_count', 'value_score']
].sort_values('value_score', ascending=False)

,customer_id,avg_order_value,total_spending,purchase_count,value_score
43,1044,434.98,14354.34,33,2.907730
56,1057,436.54,13532.74,31,2.797045
11,1012,434.50,11731.50,27,2.552443
8,1009,341.63,11615.42,34,2.524379
58,1059,287.05,8898.55,31,2.120112
9,1010,381.13,7241.47,19,1.884657
49,1050,206.60,6198.00,30,1.712847
34,1035,348.13,5918.21,17,1.657163
54,1055,319.38,6068.22,19,1.656658
3,1004,266.15,6121.45,23,1.648568


### Answer to Business Question 1


**Which customers provide the highest value to the company?**

Based on the composite value score, the top 10 customers are `1044`, `1057`, `1012`, `1009`, `1059`, `1010`, `1050`, `1035`, `1055`, and `1004`.

Customers `1044` and `1057` stand out as the strongest customers, with both very high average order values, high purchase counts, and the highest total spending in the dataset. Customer `1009` also demonstrates strong value, particularly through a high purchase frequency and total spending.

These customers represent important high value customers and could be prioritized for retention and loyalty initiatives.

### Question 2: Which customers may be at risk of not making future purchases?

In [8]:
# Calculate risk components

df['recency_risk'] = (
    (df['last_purchase_days'] - df['last_purchase_days'].min()) /
    (df['last_purchase_days'].max() - df['last_purchase_days'].min())
)

df['return_risk'] = (
    (df['returned_items'] - df['returned_items'].min()) /
    (df['returned_items'].max() - df['returned_items'].min())
)

df['satisfaction_risk'] = (
    (df['satisfaction_score'].max() - df['satisfaction_score']) /
    (df['satisfaction_score'].max() - df['satisfaction_score'].min())
)

# Calculate overall at-risk score

df['at_risk_score'] = (
    df['recency_risk'] +
    df['return_risk'] +
    df['satisfaction_risk']
)

In [9]:

fig = px.scatter(
    df,
    x='last_purchase_days',
    y='satisfaction_score',
    size='returned_items',
    color='at_risk_score',
    hover_name='customer_id',
    hover_data=[
        'last_purchase_days',
        'satisfaction_score',
        'returned_items',
        'purchase_count',
        'total_spending',
        'at_risk_score'
    ],
    color_continuous_scale='RdYlGn_r',
    title='Customer Purchase Risk',
    labels={
        'last_purchase_days': 'Days Since Last Purchase',
        'satisfaction_score': 'Satisfaction Score',
        'returned_items': 'Returned Items',
        'at_risk_score': 'At-Risk Score'
    }
)

fig.show()

In [10]:
at_risk_customers = df.nlargest(10, 'at_risk_score')[
    [
        'customer_id',
        'last_purchase_days',
        'satisfaction_score',
        'returned_items',
        'purchase_count',
        'total_spending',
        'at_risk_score'
    ]
]

at_risk_customers

,customer_id,last_purchase_days,satisfaction_score,returned_items,purchase_count,total_spending,at_risk_score
53,1054,304,1,7,9,3691.44,2.706492
17,1018,257,1,8,18,1974.60,2.701657
9,1010,276,1,7,19,7241.47,2.629144
52,1053,320,1,6,18,5805.54,2.625691
20,1021,224,1,8,16,1161.92,2.610497
7,1008,195,1,8,3,1168.74,2.530387
10,1011,272,2,8,19,2264.04,2.493094
8,1009,232,2,8,34,11615.42,2.382597
44,1045,132,1,8,24,3871.92,2.356354
36,1037,199,2,7,1,387.68,2.166436



To identify customers who may be at risk of not making future purchases, three indicators were considered: `last_purchase_days`, `satisfaction_score`, and `returned_items`.

A higher number of days since the last purchase and a higher number of returned items were considered potential risk signals, while a lower satisfaction score was also treated as an indication of higher risk.

Since these variables have different scales, each metric was normalized to a 0–1 range. The normalized risk components were then combined with equal weights to create an exploratory `at_risk_score`.

Customers with higher scores show a stronger combination of potential risk signals. This score is used to prioritize customers for further investigation and does not represent a predictive churn model.

### Answer to Business Question 2


**Which customers may be at risk of not making future purchases?**

The analysis identified customers with the strongest combination of potential risk signals. The top 10 at risk customers are **`1054`, `1018`, `1010`, `1053`, `1021`, `1008`, `1011`, `1009`, `1045`, and `1037`**.

These customers generally show a long period since their last purchase combined with low satisfaction scores and/or a higher number of returned items.

Customers `1010` and `1009` are particularly important because they appear in both the high-value and at-risk customer groups. Customer 1010 has total spending of **7,241.47** across 19 purchases, while customer 1009 has total spending of **11,615.42** across 34 purchases. Losing these customers could have a meaningful impact on the company. Therefore, high-value customers who also show signs of being at risk should receive priority in retention efforts.

### Question 3: Which cities are the most suitable for advertising?

In [11]:
customer_count_by_city = df.groupby('city')['customer_id'].count()
customer_count_by_city

city
Ahvaz       8
Isfahan     5
Karaj       7
Mashhad    11
Rasht       5
Shiraz      6
Tabriz     12
Tehran      6
Name: customer_id, dtype: int64

In [12]:
total_spending_by_city = df.groupby('city')['total_spending'].sum()
total_spending_by_city

city
Ahvaz      23384.78
Isfahan    21975.32
Karaj      26189.81
Mashhad    43197.58
Rasht      15168.54
Shiraz     20715.92
Tabriz     36192.49
Tehran     15161.29
Name: total_spending, dtype: float64

In [13]:
city_summary = df.groupby('city').agg(
    customer_count=('customer_id', 'count'),
    total_spending=('total_spending', 'sum'),
    avg_order_value=('avg_order_value', 'mean'),
    avg_spending_per_customer=('total_spending', 'mean')
).reset_index()

city_summary.sort_values('total_spending', ascending=False)

,city,customer_count,total_spending,avg_order_value,avg_spending_per_customer
3,Mashhad,11,43197.58,210.508182,3927.052727
6,Tabriz,12,36192.49,238.901667,3016.040833
2,Karaj,7,26189.81,211.440000,3741.401429
0,Ahvaz,8,23384.78,216.531250,2923.097500
1,Isfahan,5,21975.32,212.680000,4395.064000
5,Shiraz,6,20715.92,180.983333,3452.653333
4,Rasht,5,15168.54,241.554000,3033.708000
7,Tehran,6,15161.29,172.930000,2526.881667


In [14]:
fig = px.scatter(
    city_summary,
    x='customer_count',
    y='total_spending',
    size='avg_spending_per_customer',
    hover_name='city',
    text='city',
    hover_data={
        'customer_count': True,
        'total_spending': ':.2f',
        'avg_order_value': ':.2f',
        'avg_spending_per_customer': ':.2f'
    },
    title='City Performance for Advertising',
    labels={
        'customer_count': 'Number of Customers',
        'total_spending': 'Total Spending',
        'avg_spending_per_customer': 'Average Spending per Customer'
    }
)

fig.update_traces(
    textposition='top center'
)

fig.show()

To evaluate which cities may be more suitable for advertising, customer and spending patterns were analyzed at the city level.

For each city, the number of customers, total spending, average order value, and average spending per customer were calculated. A scatter plot was then created to compare the number of customers with total spending, while the size of each point represents the average spending per customer.

This analysis helps identify cities with a larger customer base, higher overall spending, and stronger customer value.

### Answer to Business Question 3

**Which cities are the most suitable for advertising?**

Based on the analysis, `Mashhad` and `Tabriz` appear to be the strongest candidates for broader advertising campaigns.

 Mashhad has the highest total spending (**43,197.58**) and a large customer base (**11 customers**), while Tabriz has the largest customer base (**12 customers**) and the second-highest total spending (**36,192.49**).

`Isfahan` and `Karaj` may also be attractive for more targeted advertising. Although they have fewer customers, their average spending per customer is relatively high, particularly in Isfahan (**4,395.06**) and Karaj (**3,741.40**).

Therefore, Mashhad and Tabriz could be prioritized for campaigns aimed at reaching a larger customer base, while Isfahan and Karaj may be considered for targeted campaigns focused on higher-value customers.

### Question 4: Are Discounts actually useful?

In [15]:
df['discount_used'].value_counts()

discount_used
No     34
Yes    26
Name: count, dtype: int64

In [16]:
discount_summary = df.groupby('discount_used').agg(
    customer_count=('customer_id', 'count'),
    total_spending=('total_spending', 'sum'),
    avg_spending_per_customer=('total_spending', 'mean'),
    avg_order_value=('avg_order_value', 'mean'),
    avg_purchase_count=('purchase_count', 'mean')
).reset_index()

discount_summary

,discount_used,customer_count,total_spending,avg_spending_per_customer,avg_order_value,avg_purchase_count
0,No,34,127049.20,3736.741176,220.700588,17.294118
1,Yes,26,74936.53,2882.174231,203.291154,17.500000


In [17]:
# Bar Plot
discount_metrics = discount_summary.melt(
    id_vars='discount_used',
    value_vars=[
        'avg_spending_per_customer',
        'avg_order_value',
        'avg_purchase_count'
    ],
    var_name='metric',
    value_name='value'
)

fig = px.bar(
    discount_metrics,
    x='metric',
    y='value',
    color='discount_used',
    barmode='group',
    text_auto='.2f',
    title='Customer Performance: Discount vs. No Discount'
)

fig.show()


To evaluate whether discounts were associated with higher customer spending, customers were divided into two groups based on `discount_used`: customers who used a discount (`Yes`) and those who did not (`No`).

The two groups were compared using customer count, total spending, average spending per customer, average order value, and average purchase count.

Because the two groups have different numbers of customers, average spending per customer and average order value were given more importance when comparing customer purchasing behavior.

### Answer to Business Question 4


**Are discounts actually effective?**

In this dataset, customers who used discounts did not show higher spending performance than customers who did not use discounts. The non-discount group had a higher average spending per customer (**3,736.74 vs. 2,882.17**) and a higher average order value (**220.70 vs. 203.29**).

The average purchase count was slightly higher among customers who used discounts (**17.50 vs. 17.29**), but the difference was small.

Therefore, the analysis does not provide evidence that discounts were associated with higher customer spending in this dataset. However, since this is an observational analysis, it cannot establish that discounts caused lower spending. Further analysis or controlled experiments would be needed to measure the causal impact of discounts.

### Question 5 : What actions should the company take to increase sales?

### Customer Value by Membership Tier

In [18]:
membership_summary = df.groupby('membership_tier').agg(
    customer_count=('customer_id', 'count'),
    total_spending=('total_spending', 'sum'),
    avg_spending_per_customer=('total_spending', 'mean'),
    avg_order_value=('avg_order_value', 'mean'),
    avg_purchase_count=('purchase_count', 'mean'),
    avg_satisfaction=('satisfaction_score', 'mean'),
    avg_last_purchase_days=('last_purchase_days', 'mean')
).reset_index()

membership_summary

,membership_tier,customer_count,total_spending,avg_spending_per_customer,avg_order_value,avg_purchase_count,avg_satisfaction,avg_last_purchase_days
0,Bronze,18,56999.81,3166.656111,203.451667,16.888889,3.111111,193.611111
1,Gold,19,75153.20,3955.431579,230.909474,17.052632,2.736842,197.421053
2,Silver,8,33439.41,4179.926250,281.532500,16.500000,2.500000,236.000000
3,VIP,15,36393.31,2426.220667,165.848000,18.866667,3.400000,186.133333



To better understand customer segments, purchasing behavior was compared across membership tiers. Average spending per customer and average order value were used as the main indicators of customer value, while the size of each point represents the number of customers in each tier.

This visualization helps identify differences in customer value and purchasing behavior across membership tiers and can support more targeted sales and retention strategies.

In [19]:
# Scatter Plot

fig = px.scatter(
    membership_summary,
    x='avg_spending_per_customer',
    y='avg_order_value',
    size='customer_count',
    color='membership_tier',
    text='membership_tier',
    hover_name='membership_tier',
    hover_data={
        'customer_count': True,
        'total_spending': ':.2f',
        'avg_spending_per_customer': ':.2f',
        'avg_order_value': ':.2f',
        'avg_purchase_count': ':.2f',
        'avg_satisfaction': ':.2f',
        'avg_last_purchase_days': ':.2f'
    },
    title='Customer Value by Membership Tier',
    labels={
        'avg_spending_per_customer': 'Average Spending per Customer',
        'avg_order_value': 'Average Order Value',
        'customer_count': 'Number of Customers'
    }
)

fig.update_traces(textposition='top center')

fig.show()

The analysis shows notable differences across membership tiers. Silver customers have the highest average spending per customer (**4,179.93**) and the highest average order value (**281.53**), despite having the smallest customer base. Gold customers generate the highest total spending (**75,153.20**), while VIP customers have the highest average purchase count (**18.87**).

These results suggest that membership tiers should not be evaluated solely by their labels. The company should consider the actual purchasing behavior of each segment when designing sales and retention strategies. 

In particular, The Silver segment may deserve further attention because it shows high customer value despite having a smaller customer base, while Gold customers represent an important source of overall revenue.

### Customer Value by Age Group


To explore whether customer value varies across different age groups, customers were grouped into five age ranges: **18–30, 31–40, 41–50, 51–60, and 61–65**.

The groups were compared based on **average spending per customer** and **average order value** to identify which age groups generate higher customer value.

In [20]:
age_summary = df.groupby(
    pd.cut(
        df['age'],
        bins=[17, 30, 40, 50, 60, 65],
        labels=['18-30', '31-40', '41-50', '51-60', '61-65']
    ),
    observed=True
).agg(
    customer_count=('customer_id', 'count'),
    total_spending=('total_spending', 'sum'),
    avg_spending_per_customer=('total_spending', 'mean'),
    avg_order_value=('avg_order_value', 'mean')
).reset_index()

In [21]:
age_summary['age'] = age_summary['age'].astype(str)

In [22]:
fig = px.bar(
    age_summary,
    x='age',
    y='avg_spending_per_customer',
    text_auto='.2f',
    title='Average Customer Spending by Age Group',
    labels={
        'age': 'Age Group',
        'avg_spending_per_customer': 'Average Spending per Customer'
    }
)

fig.show()

In [23]:
fig = px.bar(
    age_summary,
    x='age',
    y='avg_order_value',
    text_auto='.2f',
    title='Average Order Value by Age Group',
    labels={
        'age': 'Age Group',
        'avg_order_value': 'Average Order Value'
    }
)

fig.show()


Customers aged **50 and above** show noticeably higher average spending and average order values compared with younger age groups. The **61–65** group has the highest average spending per customer (**5,827.32**) and average order value (**280.53**), followed by the **51–60** group.

This suggests that older customers may represent a valuable customer segment for targeted marketing strategies. However, since the dataset is relatively small, this finding should be validated using a larger customer base before making major marketing decisions.

### Customer Value by Device and Payment Method

To explore whether purchasing behavior differs across customer channels, customers were grouped by `device` and `payment_method`. The groups were compared based on customer count, total spending, average spending per customer, and average order value.

In [24]:
channel_summary = df.groupby(['device', 'payment_method']).agg(
    customer_count=('customer_id', 'count'),
    total_spending=('total_spending', 'sum'),
    avg_spending_per_customer=('total_spending', 'mean'),
    avg_order_value=('avg_order_value', 'mean')
).reset_index()

channel_summary

,device,payment_method,customer_count,total_spending,avg_spending_per_customer,avg_order_value
0,Android,Card,9,35558.26,3950.917778,221.635556
1,Android,Cash,5,19924.07,3984.814000,199.718000
2,Android,Online_Wallet,8,20617.19,2577.148750,202.746250
3,Web,Card,6,28547.88,4757.980000,210.938333
4,Web,Cash,8,15614.96,1951.870000,223.696250
5,Web,Online_Wallet,6,13792.36,2298.726667,239.168333
6,iPhone,Card,3,6615.95,2205.316667,258.083333
7,iPhone,Cash,6,24321.98,4053.663333,144.145000
8,iPhone,Online_Wallet,9,36993.08,4110.342222,227.197778



The results show differences across device and payment method combinations. For example, Web customers using Card have the highest average spending per customer, while iPhone customers using Online Wallet generate the highest total spending among the combinations.

However, some groups contain relatively few customers. Therefore, these findings should be treated as exploratory insights and validated with a larger dataset before making major decisions.

### Answer to Business Question 5



**What actions should the company take to increase sales?**

Based on the findings from the previous analyses, several actions can be recommended:

1. **Use targeted rather than broad discount strategies.** Customers who used discounts did not show higher average spending or average order value in this dataset. The company should collect more detailed information about discounts, such as discount amount, percentage, and campaign type, and evaluate their effectiveness before expanding discount campaigns.

2. **Prioritize high-value customers who show signs of purchase risk.** Customers **1009 and 1010** appeared in both the high value and at-risk groups. These customers should be prioritized for retention efforts and personalized follow-up.

3. **Adopt city-specific advertising strategies.** **Mashhad and Tabriz** appear suitable for broader campaigns because of their larger customer bases and higher total spending, while **Isfahan and Karaj** may be suitable for more targeted campaigns focused on higher value customers.

4. **Use membership tiers and customer behavior for targeted segmentation.** The analysis shows meaningful differences across membership tiers. **Silver customers have the highest average spending per customer and average order value, while VIP customers have the highest average purchase count.** Therefore, strategies should be based on actual customer behavior rather than membership labels alone.

5. **Address customer experience issues and prioritize retention.** Customers with low satisfaction and frequent returns may require service recovery or customer experience improvements rather than simply receiving additional discounts. High value customers with high purchase risk should receive the highest retention priority, while lower-value at-risk customers can be targeted with more cost efficient strategies.

6. **Use controlled experiments to evaluate marketing strategies.** A/B testing can be used to determine whether specific discounts, offers, or campaigns actually increase purchases and revenue before applying them on a larger scale.

7. **Consider age-based targeting:** Customers aged 50 and above show higher average customer value in this dataset, making them a potential segment for targeted marketing campaigns.

8. **Optimize customer channels:** Analyze device and payment method behavior to identify high value customer channels and optimize the purchasing experience and marketing campaigns accordingly.

## Conclusion

This analysis provided several insights into customer value, purchase risk, advertising opportunities, and sales strategies.

The analysis identified high value customers who should be prioritized for retention, particularly customers 1009 and 1010, who showed both high customer value and potential purchase risk. At the city level, Mashhad and Tabriz showed strong potential for broader advertising campaigns, while Isfahan and Karaj may be suitable for more targeted campaigns.

Discounts did not show higher average spending or average order value in this dataset, suggesting that discounts should not be used as the primary sales strategy without further evaluation. More detailed discount data and controlled experiments could help determine which offers are actually effective.

The analysis also showed meaningful differences across membership tiers. Silver customers had the highest average spending per customer and average order value, while VIP customers had the highest average purchase count. This suggests that customer strategies should be based on actual purchasing behavior rather than membership labels alone.

Overall, the company can increase sales by combining customer segmentation, targeted retention, city specific advertising, improved customer experience, and data driven testing of marketing strategies.